# Spalanie jednostek obliczeniowych Colaba dla beki

Problem: sklasyfikować 7 liczb jako "atak" albo "nie atak".
`RandomForestClassifier` robi to w [ml/train.py](../ml/train.py) w
**mniej niż sekundę na CPU** z wynikiem precision=0.973/recall=1.000.

Zamiast tego: **gigantyczna, absurdalnie przewymiarowana sieć
neuronowa na najmocniejszym GPU, jakie Colab da**, trenowana bez
sensu długo, żeby zrobić dokładnie to samo, gorzej, drożej i wolniej.

**Wymaga runtime z GPU** (najlepiej A100, jeśli Twój plan Colab na to
pozwala: `Runtime -> Change runtime type -> A100 GPU`).

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "BRAK -- wlacz GPU w Runtime, bo to psuje cala zabawe")

In [ ]:
import getpass

GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token (scope: repo): ")
REPO = "pamsmediatech-lang/ai-waf-spec"
!git clone https://{GITHUB_TOKEN}@github.com/{REPO}.git /content/ai-waf-spec 2>&1 | tail -5
%cd /content/ai-waf-spec
import sys
sys.path.insert(0, "/content/ai-waf-spec")

In [ ]:
# Ten sam zbior, co dla porzadnego RandomForest -- 7 cech, ~1200 probek.
from dataclasses import fields
import numpy as np
import torch

from ml.dataset import FEATURE_NAMES, build_dataset

X_dicts, y = build_dataset(n_benign=600, n_malicious=600, seed=42)
X = np.array([[row[name] for name in FEATURE_NAMES] for row in X_dicts], dtype=np.float32)
y = np.array(y, dtype=np.int64)
print(X.shape, y.shape, "-- tak, to jest cale nasze 'big data'")

In [ ]:
# NAJGLUPSZA MOZLIWA ARCHITEKTURA: 7 wejsc -> 40 warstw po 4096 neuronow
# -> 2 wyjscia. Kilkaset milionow parametrow, zeby nauczyc sie tego, co
# jedno drzewo decyzyjne ogarnia siedmioma porownaniami "if".
import torch.nn as nn

WIDTH = 4096
DEPTH = 40

layers = [nn.Linear(len(FEATURE_NAMES), WIDTH), nn.ReLU()]
for _ in range(DEPTH):
    layers += [nn.Linear(WIDTH, WIDTH), nn.ReLU()]
layers += [nn.Linear(WIDTH, 2)]

model = nn.Sequential(*layers).cuda()
n_params = sum(p.numel() for p in model.parameters())
print(f"parametrow: {n_params:,} -- do sklasyfikowania 7 liczb")
print(f"to jest {n_params / 1200:,.0f} parametrow na kazda probke treningowa")

In [ ]:
# Batch size 1 (celowo, zeby GPU nudzilo sie miedzy krokami), zero
# early stopping, 500 epok na 1200 probkach. To dosc, zeby zauwazalnie
# nadgryzc jednostki obliczeniowe, nie dosc, zeby zrobic cokolwiek
# uzytecznego, czego RandomForest juz nie zrobil lepiej.
import time

X_t = torch.tensor(X).cuda()
y_t = torch.tensor(y).cuda()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_fn = nn.CrossEntropyLoss()

EPOCHS = 500
start = time.time()
for epoch in range(EPOCHS):
    perm = torch.randperm(len(X_t))
    total_loss = 0.0
    for i in perm:  # batch size 1, na zlosc
        optimizer.zero_grad()
        out = model(X_t[i:i+1])
        loss = loss_fn(out, y_t[i:i+1])
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if epoch % 10 == 0:
        elapsed = time.time() - start
        print(f"epoka {epoch:4d}/{EPOCHS} | strata {total_loss/len(X_t):.4f} | "
              f"minelo {elapsed:6.1f}s | jednostki obliczeniowe placza")
print(f"\nCALOSC: {time.time()-start:.1f}s, {n_params:,} parametrow, "
      f"zeby dogonic (moze) RandomForest z ml/train.py sprzed {EPOCHS} epok.")

## Rachunek strat

- `ml/train.py` (RandomForest, CPU): **< 1 sekunda**, precision 0.973, recall 1.000
- ten notebook (sieć ~500M parametrów, A100): **kilkanaście-kilkadziesiąt minut**, wynik prawdopodobnie podobny albo gorszy (500 epok batch=1 na 1200 próbkach to przepis na przeuczenie), i to za realne jednostki obliczeniowe Colaba

Zadanie wykonane: maksimum zmarnowanego kompute, zero dodatkowej wartości. 🎷🔥